In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

import os
import joblib

In [3]:
df = pd.read_csv("data/dataset.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df = df.drop(columns=["Cabin"]) # удаление столбца Cabin

In [6]:
Y = df["Survived"].copy()

In [7]:
X = df.drop(columns=["Survived", "PassengerId", "Ticket", "Name"]).copy()

In [8]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [10]:
os.makedirs("data/splits", exist_ok=True)

X_test.to_csv("data/splits/X_test.csv", index=False)
Y_test.to_csv("data/splits/Y_test.csv", index=False)


In [11]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

In [12]:
numeric_features

['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

In [13]:
categorical_features

['Sex', 'Embarked']

In [14]:
numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))  # заполняем пропуски медианой столбца
        #("scaler", StandardScaler())
    ])

categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # заполняем пропуски популярным значением
        ("onehot", OneHotEncoder(handle_unknown="ignore")),  # кодируем категориальные признаки /
        # если встретится новая категория в test — проигнорим ее
    ])

In [15]:
preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"  # удалить все колонки, которые не указаны в transformers
    )

In [16]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
    max_features="sqrt"
))
])

In [17]:
pipeline.fit(X_train, Y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
# Проверка, что модель "fitted"
rf = pipeline.named_steps.get("model", None)
if rf is None:
    raise ValueError("В pipeline нет шага 'model'. Проверь названия шагов pipeline.named_steps")

In [19]:
Y_pred = pipeline.predict(X_test)

In [20]:
metrics = {
    "Model": "RandomForest (manual tune)",
    "Accuracy": accuracy_score(Y_test, Y_pred),
    "Precision": precision_score(Y_test, Y_pred, zero_division=0),
    "Recall": recall_score(Y_test, Y_pred, zero_division=0),
    "F1": f1_score(Y_test, Y_pred, zero_division=0),
}

metrics

{'Model': 'RandomForest (manual tune)',
 'Accuracy': 0.8268156424581006,
 'Precision': 0.8166666666666667,
 'Recall': 0.7101449275362319,
 'F1': 0.7596899224806202}

In [21]:
os.makedirs("models", exist_ok=True)

joblib.dump(pipeline, "models/model.joblib")

['models/model.joblib']

In [22]:
loaded = joblib.load("models/model.joblib")

pred = loaded.predict(X_test[:5])
print("Loaded OK, sample preds:", pred)

Loaded OK, sample preds: [0 0 0 0 1]
